<a href="https://colab.research.google.com/github/antimleite/Agentes-Inteligentes/blob/main/SIMULA%C3%87%C3%83O_E_AG%C3%8ANCIA_INTELIGENTE_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ─── Instalação ───────────────────────────────────────────────────────────────
!pip install -q openai

# ─── Imports ──────────────────────────────────────────────────────────────────
import random
import json
from dataclasses import dataclass
from enum import Enum
from google.colab import userdata
from openai import OpenAI

# ─── Tipos ────────────────────────────────────────────────────────────────────

Position = tuple[int, int]

class ActionType(Enum):
    MOVE_UP    = "MOVE_UP"
    MOVE_DOWN  = "MOVE_DOWN"
    MOVE_LEFT  = "MOVE_LEFT"
    MOVE_RIGHT = "MOVE_RIGHT"
    WAIT       = "WAIT"

@dataclass(frozen=True)
class Observation:
    step: int
    self_position: Position
    visible_cells: dict[Position, str]

# ─── Ambiente ─────────────────────────────────────────────────────────────────

class WarehouseEnv:
    def __init__(self, width=10, height=5, num_packages=8):
        self.width = width
        self.height = height
        self.step_count = 0

        start_positions = {(0, 0), (width - 1, height - 1)}
        self.packages: set[Position] = set()
        while len(self.packages) < num_packages:
            pos = (random.randint(0, width - 1), random.randint(0, height - 1))
            if pos not in start_positions:
                self.packages.add(pos)

        self.agent_positions: dict[str, Position] = {
            "Supervisor": (0, 0),
            "Operario":   (width - 1, height - 1),
        }
        self.scores: dict[str, int] = {"Supervisor": 0, "Operario": 0}

    def get_observation(self, agent_name: str, radius: int = 2) -> Observation:
        pos = self.agent_positions[agent_name]
        visible: dict[Position, str] = {}
        for dy in range(-radius, radius + 1):
            for dx in range(-radius, radius + 1):
                x, y = pos[0] + dx, pos[1] + dy
                if 0 <= x < self.width and 0 <= y < self.height:
                    visible[(x, y)] = "package" if (x, y) in self.packages else "empty"
        return Observation(self.step_count, pos, visible)

    def apply_action(self, agent_name: str, action: ActionType):
        deltas = {
            ActionType.MOVE_UP:    (0, -1),
            ActionType.MOVE_DOWN:  (0,  1),
            ActionType.MOVE_LEFT:  (-1, 0),
            ActionType.MOVE_RIGHT: (1,  0),
            ActionType.WAIT:       (0,  0),
        }
        pos = self.agent_positions[agent_name]
        dx, dy = deltas[action]
        new_pos = (pos[0] + dx, pos[1] + dy)

        if 0 <= new_pos[0] < self.width and 0 <= new_pos[1] < self.height:
            self.agent_positions[agent_name] = new_pos
            if new_pos in self.packages:
                self.packages.remove(new_pos)
                self.scores[agent_name] += 10

    def render(self):
        grid = [["." for _ in range(self.width)] for _ in range(self.height)]
        for x, y in self.packages:
            grid[y][x] = "*"
        sx, sy = self.agent_positions["Supervisor"]
        ox, oy = self.agent_positions["Operario"]
        if (sx, sy) == (ox, oy):
            grid[sy][sx] = "X"
        else:
            grid[sy][sx] = "S"
            grid[oy][ox] = "O"
        print("\n+" + "-" * self.width + "+")
        for row in grid:
            print("|" + "".join(row) + "|")
        print("+" + "-" * self.width + "+")
        print(f"Step {self.step_count} | Scores: {self.scores} | Pacotes: {len(self.packages)}")

# ─── Agente 1: LLM — OpenAI (GPT-4o-mini) ───────────────────────────────────

class LLMSupervisorAgent:
    SYSTEM_PROMPT = """Você é o Supervisor de um armazém. Seu objetivo é coletar pacotes (*).
Responda APENAS com a ação: MOVE_UP, MOVE_DOWN, MOVE_LEFT, MOVE_RIGHT ou WAIT.
Não escreva mais nada além de uma dessas palavras."""

    def __init__(self):
        try:
            self.api_key = userdata.get("Agentes_Inteligentes")
            self.client = OpenAI(api_key=self.api_key)
        except Exception as e:
            print(f"⚠️ Erro ao acessar a chave: {e}")
            self.client = None

    def decide(self, obs: Observation) -> ActionType:
        if not self.client:
            return ActionType.WAIT

        packages_visible = [str(p) for p, v in obs.visible_cells.items() if v == "package"]
        user_msg = (
            f"Posição: {obs.self_position}\n"
            f"Pacotes visíveis em: {packages_visible or 'nenhum'}\n"
            "Ação:"
        )

        try:
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": self.SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg}
                ],
                max_tokens=5,
                temperature=0
            )
            action_str = response.choices[0].message.content.strip().upper()

            for action in ActionType:
                if action.value in action_str:
                    return action
            return ActionType.WAIT

        except Exception as e:
            print(f"[OpenAI fallback] {e}")
            return ActionType.WAIT

# ─── Agente 2: Reflexo Simples ───────────────────────────────────────────────

class SimpleReflexOperarioAgent:
    def decide(self, obs: Observation) -> ActionType:
        pos = obs.self_position
        packages = [(p, v) for p, v in obs.visible_cells.items() if v == "package"]
        if packages:
            target = min(packages, key=lambda pv: abs(pv[0][0] - pos[0]) + abs(pv[0][1] - pos[1]))[0]
            dx = target[0] - pos[0]
            dy = target[1] - pos[1]
            if abs(dx) >= abs(dy):
                return ActionType.MOVE_RIGHT if dx > 0 else ActionType.MOVE_LEFT
            else:
                return ActionType.MOVE_DOWN if dy > 0 else ActionType.MOVE_UP
        return random.choice(list(ActionType))

# ─── Loop Principal ───────────────────────────────────────────────────────────

def run_simulation(max_steps: int = 20):
    env = WarehouseEnv(width=10, height=5, num_packages=8)
    supervisor = LLMSupervisorAgent()
    operario   = SimpleReflexOperarioAgent()

    print("=== Simulação iCEV 2026: Agentes Inteligentes ===")
    print("Supervisor: OpenAI LLM | Operário: Reflexo Simples")
    env.render()

    for step in range(max_steps):
        env.step_count = step + 1
        if not env.packages:
            print("\n✅ Todos os pacotes coletados!")
            break

        act_s = supervisor.decide(env.get_observation("Supervisor"))
        env.apply_action("Supervisor", act_s)

        act_o = operario.decide(env.get_observation("Operario"))
        env.apply_action("Operario", act_o)

        print(f"\n[Passo {step+1}] S -> {act_s.value} | O -> {act_o.value}")
        env.render()

    print(f"\n=== Resultado Final ===")
    print(f"Supervisor (LLM): {env.scores['Supervisor']} pts")
    print(f"Operário (Reflexo): {env.scores['Operario']} pts")

# ─── Executar ─────────────────────────────────────────────────────────────────
run_simulation(max_steps=20)

=== Simulação iCEV 2026: Agentes Inteligentes ===
Supervisor: OpenAI LLM | Operário: Reflexo Simples

+----------+
|S.........|
|.....*....|
|....*.**..|
|..*...*.*.|
|........*O|
+----------+
Step 0 | Scores: {'Supervisor': 0, 'Operario': 0} | Pacotes: 8

[Passo 1] S -> WAIT | O -> MOVE_LEFT

+----------+
|S.........|
|.....*....|
|....*.**..|
|..*...*.*.|
|........O.|
+----------+
Step 1 | Scores: {'Supervisor': 0, 'Operario': 10} | Pacotes: 7

[Passo 2] S -> WAIT | O -> MOVE_UP

+----------+
|S.........|
|.....*....|
|....*.**..|
|..*...*.O.|
|..........|
+----------+
Step 2 | Scores: {'Supervisor': 0, 'Operario': 20} | Pacotes: 6

[Passo 3] S -> WAIT | O -> MOVE_LEFT

+----------+
|S.........|
|.....*....|
|....*.**..|
|..*...*O..|
|..........|
+----------+
Step 3 | Scores: {'Supervisor': 0, 'Operario': 20} | Pacotes: 6

[Passo 4] S -> WAIT | O -> MOVE_UP

+----------+
|S.........|
|.....*....|
|....*.*O..|
|..*...*...|
|..........|
+----------+
Step 4 | Scores: {'Supervisor': 0, '